# 65에폭과 95에폭에서 원판 궤적은 얼마나 달라질까?

원본 `yolo26n.pt`에서 100에폭을 한 번 학습하고, 65번째와 95번째 에폭이 끝난 순간의 가중치를 같은 검증 영상에 적용합니다. 학습은 두 번째 코드 셀에서 직접 시작합니다. 중간 가중치와 평가 결과는 이번 실행 폴더에 저장됩니다.

## 1. 시작 가중치와 분할 확인

기존 베이스라인의 `args.yaml`을 읽습니다. 100에폭, patience 100, 사람별 분할과 증강을 포함한 설정이 그대로인지 확인하세요. 이번에 바꾸는 것은 중간 가중치 저장뿐입니다.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

from wlcv.config import VIDEOS
from wlcv.data.frames import read_rgb
from wlcv.evaluation.epoch_pair import summarize_epochs
from wlcv.evaluation.schedule_review import compare_runs, evaluate_motion, review_inputs
from wlcv.training.plate_epoch_pair import RUN, completed_weights, train_epoch_pair
from wlcv.training.plate_schedule import baseline_settings

settings = baseline_settings()
data = Path(settings["data"])
videos, _, _ = review_inputs(data)
for video in videos:
    (VIDEOS / f"{video}.mp4").resolve(strict=True)
display(pd.Series({key: settings[key] for key in
                   ["model", "epochs", "patience", "device", "batch", "imgsz", "seed", "optimizer"]},
                  name="기존 베이스라인 설정").to_frame())
print("검증 영상", ", ".join(videos))
print("이번 실행 폴더", RUN)

## 2. Mac에서 학습 시작

준비되면 `START_TRAINING`을 `True`로 바꾸세요. 첫 실행은 사전학습 가중치에서 시작하고, 65·95에폭에는 해당 시점의 모델을 각각 저장합니다. 커널이 중단됐다면 첫 셀부터 다시 실행하세요. 저장된 `last.pt`가 있으면 이어서 학습하고, 완료한 실행이면 저장 파일을 확인합니다.

In [ ]:
START_TRAINING = False

if START_TRAINING:
    for epoch, path in train_epoch_pair().items():
        print(f"{epoch}에폭 확인 · {path}")
else:
    print("학습 대기 · 학습을 마쳤다면 다음 셀부터 결과를 확인하세요.")

## 3. 같은 동작 프레임에서 두 가중치 평가

검증 영상 6편을 각각 처음부터 읽습니다. 수동 원판 박스가 있는 ready~finish 구간만 채점하고, finish가 없는 하프 영상은 마지막으로 표시한 동작까지 봅니다. 직접 검출은 신뢰도 0.10, 시간 연결은 기존 규칙(후보 0.001, 반지름은 직전 박스 폭의 0.5배, 최대 2프레임)을 사용합니다. 선택된 박스가 없거나 정답과 IoU가 0.50 미만이면 미탐 한 장입니다.

In [ ]:
pair = None
summary = None
history = RUN / "results.csv"
output = RUN / "comparison"

if history.is_file() and pd.read_csv(history).epoch.iloc[-1] == 100:
    weights = completed_weights()
    records = {}
    for epoch, path in tqdm(weights.items(), desc="가중치 비교", unit="개"):
        records[epoch] = evaluate_motion(path, data)
    pair = compare_runs(records[65], records[95], labels=("65", "95"))
    summary = summarize_epochs(pair)
    output.mkdir(exist_ok=True)
    pair.to_csv(output / "frames.csv", index=False)
    summary.to_csv(output / "summary.csv", index=False)
    display(summary[summary["영상"] == "전체"].round(2))
    print("결과 폴더", output)
else:
    print("100에폭 학습이 끝나면 이 셀을 실행하세요.")

## 4. 선수별 미탐과 허브 오차 비교

두 박스가 모두 정탐이고 허브를 **직접 판독한 같은 프레임**에서만 위치 오차를 계산합니다. 세로 오차는 ‘모델 박스 중심 y − 수동 허브 y’이며 양수면 모델 중심이 더 아래입니다. 평균 절대 오차는 거리의 크기, 평균 편향은 위·아래로 치우친 방향을 나타냅니다. 공통 정탐 허브가 0개인 영상의 오차는 빈칸입니다.

표에는 두 방식이 모두 있습니다. 그래프는 아래 `method`로 고른 방식을 보여줍니다. 전체 평균은 평가 가능한 프레임 단위이므로 선수마다 기여하는 장수가 다릅니다.

In [ ]:
method = "시간 연결 적용"  # "직접 검출"로 바꾸어 다시 볼 수 있습니다.

if summary is not None:
    per_video = summary[summary["영상"] != "전체"]
    display(per_video.round(2))
    shown = per_video[per_video["방식"] == method]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    for ax, column, title in zip(
        axes, ["미탐", "허브 세로 평균 절대 오차(px)"], ["Missed plate frames", "Paired hub vertical MAE (px)"]
    ):
        wide = shown.pivot(index="영상", columns="에폭", values=column)
        wide.index = [video.split("_")[0] for video in wide.index]
        # 점이 없는 선수는 측정값이 없다. 오차가 0인 선수는 0에 점이 찍힌다.
        for epoch, marker in [(65, "o"), (95, "x")]:
            ax.plot(wide.index, wide[epoch], marker=marker, linestyle="none", markersize=8, label=f"Epoch {epoch}")
        ax.set_title(title)
        ax.set_ylim(bottom=-0.2)
        ax.legend()
    plt.show()

## 5. 판정이 달라진 프레임과 원본 보기

‘복구’는 65에서 놓치고 95에서 맞힌 장면, ‘손실’은 반대입니다. 아래 셀은 선택한 방식에서 바뀐 첫 장면을 보여줍니다. 바뀐 판정이 없으면 공통 정탐 한 장을 표시합니다. `video_id`와 `frame_index`에 검증 영상 ID와 0부터 센 프레임 번호를 지정해 다른 장면도 확인할 수 있습니다.

In [ ]:
from matplotlib.patches import Rectangle

video_id = None
frame_index = None

if pair is not None:
    method_id = {"시간 연결 적용": "Past link", "직접 검출": "YOLO"}[method]
    rows = pair[pair.method == method_id].sort_values(["video_id", "frame_index"])
    changed = rows[rows.change.isin(["복구", "손실"])]
    columns = ["video_id", "frame_index", "change", "source_65", "iou_65", "dy_px_65",
               "source_95", "iou_95", "dy_px_95"]
    display(changed[columns].round(2))
    if video_id is not None or frame_index is not None:
        example = rows[(rows.video_id == video_id) & (rows.frame_index == frame_index)]
    elif not changed.empty:
        example = changed.head(1)
    else:
        example = rows[rows["change"] == "모두 정탐"].head(1)
    if example.empty:
        print("표에서 영상 ID와 동작 중 평가 프레임 번호를 함께 선택하세요.")
    else:
        row = example.iloc[0]
        rgb = read_rgb(VIDEOS / f"{row.video_id}.mp4", int(row.frame_index))
        fig, axes = plt.subplots(1, 2, figsize=(8, 5), layout="constrained")
        for ax, epoch in zip(axes, (65, 95)):
            ax.imshow(rgb)
            x, y = row.gt_x1, row.gt_y1
            ax.add_patch(Rectangle((x, y), row.gt_x2-x, row.gt_y2-y, fill=False, color="cyan", label="Manual box"))
            if row.visibility == "visible" and pd.notna(row.manual_y):
                ax.plot(row.manual_x, row.manual_y, "+", color="cyan", label="Manual hub", markersize=12)
            x, y, right, bottom = [row[f"{side}_{epoch}"] for side in ("x1", "y1", "x2", "y2")]
            if pd.notna(x):
                ax.add_patch(Rectangle((x, y), right-x, bottom-y, fill=False, color="orange", linestyle="--",
                                       label="Selected box"))
                ax.plot((x+right)/2, (y+bottom)/2, "x", color="orange", label="Box center")
            ax.set_title(f"Epoch {epoch} | {row[f'source_{epoch}']}")
            ax.legend(fontsize=8)
            ax.axis("off")
        fig.suptitle(f"{row.video_id} | frame {row.frame_index}")
        plt.show()

미탐이 줄었는지, 같은 허브를 기준으로 세로 오차도 줄었는지 함께 보세요. 특히 a03·b02에만 차이가 몰리는지 선수별 표를 확인하세요. 이 비교가 보여주는 범위는 반복해서 본 검증 6명입니다.